In [ ]:
import sys
import subprocess
import pkgutil

def try_install(pkg_spec):
    """Try to pip install a package; return True on success."""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_spec])
        return True
    except Exception as e:
        print(f"Could not install {pkg_spec}: {e}")
        return False

# Try to install the GitHub ID3 library
installed_id3 = False
if not pkgutil.find_loader('id3') and not pkgutil.find_loader('decision_tree'):
    print('Trying to install decision-tree-id3 from GitHub...')
    installed_id3 = try_install('git+https://github.com/svaante/decision-tree-id3')
else:
    installed_id3 = True

print('ID3 lib installed:', installed_id3)
print('Proceeding — if the library is unavailable we will use sklearn DecisionTreeClassifier (criterion="entropy").')

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier, export_text
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

# Utility: safe DecisionTree using ID3 lib if available else sklearn
USE_ID3_LIB = False
try:
    import id3
    USE_ID3_LIB = True
    print('Using id3 library from GitHub.')
except Exception:
    try:
        # some forks might expose different module names; attempt import
        import decision_tree
        USE_ID3_LIB = True
        print('Using decision_tree (alternative import).')
    except Exception:
        USE_ID3_LIB = False
        print('ID3 library not available; falling back to sklearn DecisionTreeClassifier(criterion="entropy").')



In [ ]:
# 1) Car Evaluation UCI
car_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data'
col_names = ['buying','maint','doors','persons','lug_boot','safety','class']
try:
    car = pd.read_csv(car_url, header=None, names=col_names)
    print('Car dataset loaded, shape =', car.shape)
except Exception as e:
    print('Could not download dataset automatically. Provide car.data locally or check internet connection:', e)
    car = pd.DataFrame()
car.head()


In [ ]:
# 2) Label encoding ( categorical )
def encode_df(df):
    encoders = {}
    df_enc = df.copy()
    for col in df_enc.columns:
        le = LabelEncoder()
        df_enc[col] = le.fit_transform(df_enc[col].astype(str))
        encoders[col] = le
    return df_enc, encoders

if not car.empty:
    car_enc, car_encoders = encode_df(car)
    display(car_enc.head())
else:
    print('Car dataframe is empty — cannot encode.')


In [ ]:
# 3)  devide datas 80/20
if not car.empty:
    X = car_enc.drop('class', axis=1)
    y = car_enc['class']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
else:
    print('Skipping split because car dataset not available.')


In [ ]:
# 4) model train
def train_id3_like(X_tr, y_tr):
    if USE_ID3_LIB:
        # Example: interface may vary between libraries; this is a best-effort placeholder.
        try:
            # try a typical usage from some ID3 forks
            from id3 import Id3Estimator
            clf = Id3Estimator()
            clf.fit(X_tr.values.tolist(), y_tr.values.tolist())
            return clf, 'id3'
        except Exception:
            pass
    # fallback to sklearn
    clf = DecisionTreeClassifier(criterion='entropy', random_state=42)
    clf.fit(X_tr, y_tr)
    return clf, 'sklearn'

if not car.empty:
    clf_car, clf_type = train_id3_like(X_train, y_train)
    print('Trained classifier type:', clf_type)
else:
    print('No model trained — dataset missing.')


In [ ]:
# 5) Accuracy & Confusion Matrix :
if not car.empty:
    if clf_type == 'sklearn':
        y_pred = clf_car.predict(X_test)
    else:
        # try to predict using id3 library API (best-effort)
        try:
            y_pred = clf_car.predict(X_test.values.tolist())
        except Exception:
            y_pred = clf_car.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    print('Accuracy (Car, 80/20):', acc)
    print('\nConfusion Matrix:')
    print(cm)
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred))

    # plot confusion matrix
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title('Car: Confusion Matrix (80/20)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
else:
    print('Skipping evaluation — dataset missing.')


In [ ]:
# Reduced Error Pruning sklearn
from copy import deepcopy
def reduced_error_pruning_sklearn(clf, X_train, y_train, X_val, y_val):
    """A simple, slow reduced-error pruning for sklearn DecisionTreeClassifier.
    It will attempt to prune non-leaf nodes by turning them into leaves if validation accuracy
    does not decrease. Works by examining tree structure via clf.tree_.
    Returns a pruned clone of the classifier."""
    pruned = deepcopy(clf)
    best_score = accuracy_score(y_val, pruned.predict(X_val))
    changed = True
    n_iter = 0
    while changed:
        changed = False
        n_iter += 1
        tree = pruned.tree_
        # gather candidate internal node ids (those with children)
        candidates = [i for i in range(tree.node_count) if tree.children_left[i] != tree.children_right[i]]
        # Try pruning each candidate and keep the first that improves/doesn't worsen val score
        for node in candidates:
            # make a copy and prune node -> turn into leaf
            temp = deepcopy(pruned)
            # set the node to a leaf by setting children to -1 and value to majority class in that node
            # NOTE: accessing and mutating sklearn internal structures is not officially supported
            # but works in many versions for experimentation
            value = tree.value[node]
            # determine majority class
            maj_class = np.argmax(value.sum(axis=0))
            # apply changes on temp
            temp.tree_.children_left[node] = -1
            temp.tree_.children_right[node] = -1
            # set feature to -2 (unused) and threshold to -2 (unused)
            temp.tree_.feature[node] = -2
            temp.tree_.threshold[node] = -2
            # set value to zero everywhere then set majority class count to sum
            new_val = np.zeros_like(temp.tree_.value[node])
            new_val[0, maj_class] = tree.n_node_samples[node]
            temp.tree_.value[node] = new_val
            # evaluate
            try:
                score = accuracy_score(y_val, temp.predict(X_val))
            except Exception:
                score = -1
            if score >= best_score:
                pruned = temp
                best_score = score
                changed = True
                break
        if n_iter > 50:
            break
    return pruned

if not car.empty:
    # split into 70/10/20
    X_temp, X_test2, y_temp, y_test2 = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
    X_train2, X_val2, y_train2, y_val2 = train_test_split(X_temp, y_temp, test_size=0.125, random_state=1, stratify=y_temp)  # 0.125 of 0.8 = 0.1 overall
    print('Shapes (train/val/test):', X_train2.shape, X_val2.shape, X_test2.shape)
    # train full tree
    base_clf = DecisionTreeClassifier(criterion='entropy', random_state=42)
    base_clf.fit(X_train2, y_train2)
    print('Base val acc:', accuracy_score(y_val2, base_clf.predict(X_val2)))
    pruned_clf = reduced_error_pruning_sklearn(base_clf, X_train2, y_train2, X_val2, y_val2)
    print('Pruned val acc:', accuracy_score(y_val2, pruned_clf.predict(X_val2)))
    print('Test acc base:', accuracy_score(y_test2, base_clf.predict(X_test2)))
    print('Test acc pruned:', accuracy_score(y_test2, pruned_clf.predict(X_test2)))
else:
    print('Skipping pruning — dataset missing.')


In [ ]:
# 1) Nursery
nursery_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/nursery/nursery.data'
nursery_cols = ['parents','has_nurs','form','children','housing','finance','social','health','class']
try:
    nursery = pd.read_csv(nursery_url, header=None, names=nursery_cols)
    print('Nursery loaded, shape =', nursery.shape)
except Exception as e:
    print('Could not download nursery dataset automatically:', e)
    nursery = pd.DataFrame()
nursery.head()


In [ ]:
if not nursery.empty:
    nursery_enc, nursery_encoders = encode_df(nursery)
    display(nursery_enc.head())
else:
    print('Nursery dataset missing — skipping encoding.')


In [ ]:
# 2) Three devide ratio
splits = [(0.8,0.2),(0.6,0.4),(0.4,0.6)]
    
    
if not nursery.empty:
    Xn = nursery_enc.drop('class', axis=1)
    yn = nursery_enc['class']
    results = []
    for tr, te in splits:
        X_tr, X_te, y_tr, y_te = train_test_split(Xn, yn, test_size=te, train_size=tr, random_state=42, stratify=yn)
        clf = DecisionTreeClassifier(criterion='entropy', random_state=42)
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)
        acc = accuracy_score(y_te, y_pred)
        results.append({'split':f'{int(tr*100)}/{int(te*100)}','accuracy':acc})
    res_df = pd.DataFrame(results)
    display(res_df)
else:
    print('Skipping split experiments — nursery missing.')


In [ ]:
# 3) GridSearchCV (k=5)
if not nursery.empty:
    param_grid = {
        'max_depth': [None, 5, 10, 15],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 5],
        'ccp_alpha': [0.0, 0.001, 0.01]
    }
    base = DecisionTreeClassifier(criterion='entropy', random_state=42)
    grid = GridSearchCV(base, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(Xn, yn)
    print('Best params:', grid.best_params_)
    print('Best CV accuracy:', grid.best_score_)
    best_clf = grid.best_estimator_
    # evaluate with a 80/20 split to compare
    X_tr2, X_te2, y_tr2, y_te2 = train_test_split(Xn, yn, test_size=0.2, random_state=42, stratify=yn)
    best_clf.fit(X_tr2, y_tr2)
    y_pred2 = best_clf.predict(X_te2)
    print('Test accuracy with best params:', accuracy_score(y_te2, y_pred2))
else:
    print('Skipping GridSearch — nursery missing.')
